# E13 — Bidirectional semantic evidence
[Frozen protocol and numerical predictions](./E13_PROTOCOL.md), committed before execution (26c9650). CPU only. E1–E9 remain unchanged.

```
immutable evidence A --> uncertain relation A --+
                                                +--> composed mechanic C
immutable evidence B --> uncertain relation B --+          |
                         ^                                 |
                         +---- new C evidence revises ------+
```

Keep an independent C hypothesis as well. A connection must earn posterior probability, not be assumed. This operationalizes a limited interpretation of jñāpaka (reject clues with independent explanations), anvaya-vyatireka (contrast), apoddhara (abstract recurrent meaning), and anvitabhidhana (contextual meaning). The traditions disagree; see [Patanjali's inferential practice](https://www.academia.edu/37509541/From_Commentary_to_paribh%C4%81%E1%B9%A3%C4%81s_K%C4%81ty%C4%81yana_and_Pata%C3%B1jali_vis_%C3%A0_vis_Vy%C4%81%E1%B8%8Di), [Bhartrhari 3.14.75](https://www.wisdomlib.org/hinduism/book/vakyapadiya-of-bhartrihari/d/doc1337577.html), [Prabhakara/Shalikanatha](https://plato.stanford.edu/entries/literal-nonliteral-india/) and [neural relational inference](https://proceedings.mlr.press/v80/kipf18a.html).
Not a new Bayesian algorithm, not an ARC result. The hypothesis language is supplied, not invented. Binary next-state effects on static scenes, not a complete sequential world model.

### 1. Candidate semantics
Prediction before run: 18 predicates, 972 ordered compositions. No semantic labels are observed. The evaluator has truth tables solely for generating and scoring scenes. Commutative duplicates remain in the prior; this is a prior over syntactic descriptions.

In [ ]:
import os
os.environ['OPENBLAS_NUM_THREADS']='1'
os.environ['OMP_NUM_THREADS']='1'
import numpy as np
from itertools import product
from time import perf_counter

X=np.array(list(product([0,1],repeat=6)),dtype=bool)
P=np.array([X[:,i]==X[:,j] for i in range(3) for j in range(3,6)]+[X[:,i]&X[:,j] for i in range(3) for j in range(3,6)])
K=len(P); I,J,O=np.indices((K,K,3)).reshape(3,-1)
H=np.stack([P[I]&P[J],P[I]|P[J],P[I]^P[J]])[O,np.arange(len(I))].astype(float)
F=P.astype(float); EPS=.1
assert P.shape==(18,64) and H.shape==(972,64)

### 2. Exact inference
Prediction: normalized posterior; old and new concepts are jointly constrained by observed outcomes. Forward joint prediction is exactly ordinary source-informed Bayesian transfer, not evidence of an extra recursive mechanism. The mixture's independent branch leaves A and B unchanged.

In [ ]:
def likelihood(bank,q,y):
    return np.where(bank[:,q]==y,1-EPS,EPS).prod(1)

def norm(w):return w/w.sum()

def predict(a,b,l):
    w=a[I]*b[J]*l/3
    z=w.sum();w/=z
    return z,w@H, np.bincount(I,w,minlength=K)@F, np.bincount(J,w,minlength=K)@F

u=np.full(K,1/K)
z,_,pa,pb=predict(u,u,np.ones(len(H)))
assert np.isclose(z,1) and np.allclose(pa,u@F) and np.allclose(pb,u@F)

### 3. Causal episode
Prediction: source observations help predict C on structured families; C evidence improves predictions of A/B on new scenes. The independent branch should limit harmful transfer. All arms see the same disjoint observations. The target-only arm intentionally ignores A/B; the joint arm is the strong full-history comparator. The independent family samples C's latent predicates separately. Noise flips observed outcomes with known probability .1. Untouched test truth is noiseless.

In [ ]:
def run_world(rng,family):
    if family==0: i,j=rng.integers(0,9,2)
    elif family==1: i,j=rng.integers(9,18,2)
    else:i,j=rng.integers(K,size=2)
    ci,cj=(i,j) if family<2 else rng.integers(K,size=2)
    cidx=(ci*K+cj)*3+rng.integers(3)
    ta,tb,tc=F[i],F[j],H[cidx]
    q=rng.permutation(64); test=q[:16]; qa,qb,qc=q[16:19],q[19:22],q[22:30]
    ya=ta[qa].astype(bool)^(rng.random(3)<EPS); yb=tb[qb].astype(bool)^(rng.random(3)<EPS); yc=tc[qc].astype(bool)^(rng.random(8)<EPS)
    a,b=norm(likelihood(F,qa,ya)),norm(likelihood(F,qb,yb))
    u=np.full(K,1/K); ma=np.eye(K)[a.argmax()];mb=np.eye(K)[b.argmax()]
    old=np.stack([a@F,b@F]); true=np.stack([ta,tb])[:,test]
    rows=[]
    for n in (0,1,2,4,8):
        l=likelihood(H,qc[:n],yc[:n]); z0,p0,*_=predict(u,u,l); _,pm,*_=predict(ma,mb,l)
        z,p,pa,pb=predict(a,b,l); alpha=z/(z+z0)
        robust=alpha*p+(1-alpha)*p0
        source=alpha*np.stack([pa,pb])+(1-alpha)*old
        def metrics(pred,t):return [np.mean((pred>=.5)==t),np.mean((pred-t)**2)]
        rows.append([metrics(x[test],tc[test]) for x in (p0,pm,p,robust)]+[metrics(x[:,test],true) for x in (old,np.stack([pa,pb]),source)])
    return np.array(rows)

### 4. Frozen evaluation
Pre-run predictions: forward improvement at 2 exposures +8 pp for each structured family; backward improvement at 4 exposures +3 pp for each. Unrelated forward improvement at 4: -1 pp. Gate: >=5 pp forward and >=2 pp backward for both structured families, unrelated harm <=2 pp; intervals exclude zero for gains. Seeds and budget fixed in protocol. No GPU even if the gate passes: ordinary exact inference is not defeated.

In [ ]:
def main():
    t=perf_counter(); allrows=[]
    for family in range(3):
        results=[]
        for seed in (8171,9239):
            rng=np.random.default_rng(seed);r=np.array([run_world(rng,family) for _ in range(200)]);results.append(r)
        r=np.concatenate(results);allrows.append(r)
        print('FAMILY',family,'accuracy rows=0,1,2,4,8; cols target-only, MAP, joint, robust, frozen-source, joint-source, robust-source')
        print(np.round(r[:,:,:,0].mean(0)*100,2))
        print('Brier');print(np.round(r[:,:,:,1].mean(0),4))
        for name,d in [('forward@2',r[:,2,3,0]-r[:,2,0,0]),('backward@4',r[:,3,6,0]-r[:,3,4,0]),('forward@4',r[:,3,3,0]-r[:,3,0,0])]:
            boot=np.random.default_rng(928).choice(d,(3000,len(d)),replace=True).mean(1)
            print(name,round(100*d.mean(),3),np.round(100*np.quantile(boot,[.025,.975]),3))
    print('seconds',perf_counter()-t)
    return np.array(allrows)

results=main()
np.save('e13_results.npy',results)

### 5. Captured standalone CPU results (not notebook execution counts)
Same evaluation code, 1,200 worlds, 1.515 seconds; cProfile rerun 2.222 seconds including imports.

| Family | Robust forward @2 pp (95% CI) | Robust backward @4 pp (95% CI) | Robust forward @4 pp |
|---|---:|---:|---:|
| Equality | 5.156 [4.016, 6.359] | 4.742 [3.797, 5.734] | 7.109 |
| Conjunction | 1.672 [0.922, 2.422] | 1.516 [0.922, 2.094] | 2.797 |
| Independent | -0.844 [-1.766, 0.047] | -0.648 [-1.352, 0.062] | -1.422 [-2.391,-0.453] |

At 4 C exposures, equality: target-only 63.64%, MAP 67.27%, exact joint 72.28%, robust 70.75%; old frozen 67.52%, exact joint revised 73.56%, robust revised 72.26%.
Conjunction: target-only 74.41%, MAP 59.13%, exact joint 77.91%, robust 77.20%; old frozen 79.66%, exact joint revised 81.45%, robust revised 81.18%.
Independent: target-only 68.83%, MAP 55.94%, exact joint 65.47%, robust 67.41%; old frozen 73.43%, exact joint revised 71.11%, robust revised 72.78%.

The gate fails on conjunction. Positive transfer is conditional on shared structure. No improvement over exact joint inference on the matched worlds. Robust mixture reduces but does not eliminate negative transfer. Floating point ties around .5 can produce tiny zero-exposure accuracy discrepancies; Brier is unaffected. The hypothesis prior is uniform over syntactic expressions, not semantically unique functions. Both structured families use supplied predicates and the same compositional generator, not independent real domains.


### 6. Profile and decision
Standalone cProfile: 1,200 run_world calls took 1.901s cumulative; 42,000 metric calls 0.839s; 18,000 predict calls 0.533s. Evaluation overhead, not inference kernels, dominates this toy. No GPU speedup claim. The candidate table is ~486 KiB (972×64×8 bytes). Exhaustive enumeration scales poorly as arity increases. Real WARC GPU throughput cannot be inferred from this table.

Next experiment: keep uncertain semantics and independent fallback; learn new parameterized relations by contrasting complete transitions and testing whether they become useful operands in **unseen** dynamics. Compare against exact full-history inference in the same language and against learned abstraction/predicate invention. Pre-register held-out compositions, confounders and total budgets. Keep WARC pretrained prediction and TTT as the real-data baseline; benchmark official MechEval only with its official protocol. E13 does not justify a GPU run.